# Exercise 1 - Jupyter Notebook 
## Generate time series forecasts on-the-fly using using __SAP-RPT-TS__ from SAP HANA Cloud
SAP TechEd 2026, Hands-On Workshop: DA262 - Get hands-on with tabular AI- and forecasting-models in SAP HANA Cloud
<br><br>

In this exercise you will use **SAP-RPT-TS** — SAP's new __tabular AI foundation model__ specifically targeted for __time series forecasting__ —
to generate bike sales forecasts directly inside SAP HANA Cloud.

**SAP-RPT-TS** is a Relational Pre-trained Transformer model, developed and maintained by SAP, and extends the SAP-RPT-1 model family from static tabular data to temporal, sequential data.
It's key features include:
- **Pretrained Model**: Removes the need to collect training datasets, manage compute infrastructure, or wait weeks for model training cycles. Deploy immediately with production-ready accuracy across any forecast horizon.
- **Fast and Lightweight**: A compact, efficient architecture delivers low-latency forecasts and runs comfortably on standard CPUs, making it cost-effective to consume and scale across many time series.
- **In-Context Learning**: Accepts a series' recent history at inference time to return instant forecasts, without setup or per-series model building.
- **Flexible Data Structures**: Handles time series of different shapes — univariate and multivariate series, multiple targets, and both past and known-future covariates — letting you incorporate related series and external drivers to improve accuracy wherever related data is available.
- **Probabilistic and Grounded**: Produces forecasts grounded in the provided history, returning calibrated prediction intervals — not just point estimates — reducing the risk of costly over- or under-prediction decisions.
- **Available In-Database or AI Core**: The model can flexibly be consumed directly within SAP HANA Cloud co-located to application business data,  or [SAP-RPT-TS can be consumed vis SAP AI Core](https://help.sap.com/docs/sap-ai-core/generative-ai/sap-rpt-ts).

<br>

In this exercise, we will __consume SAP-RPT-TS directly from within SAP HANA Cloud__
- to access and consume the model, the HANA Cloud instance configuration requires to have the __NLP services__  enabled
- with the 2026 Q3 HANA Cloud release, usage of SAP-RPT-TS is unlocked with the [**TSFM operator** for PAL **AutoML / Pipeline models**](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-predictive-analysis-library/tsfm) ([TSFM in hana_ml 2.30](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/2026_3_QRC/en-US/pal/topics/pipeline_operator_tsfm.html))
  - **TSFM** is a pipeline operator node for __zero-shot time series forecasting__. It joins the PAL Pipeline framework alongside classic time-series operators (ARIMA, BSTS, AMTSA, HGBT-TS, MLR-TS, …) and can be selected by AutoML or invoked directly from Pipeline SQL functions.
- __Zero-shot time series forecasting__ implies here 
  - **No model training step**, if used `PAL_PIPELINE_FIT` snapshots the historical `[ID, TARGET, COV...]` table into the model. No gradient   descent, no epochs.
  - **All actual inference happens at predict time.** `PAL_PIPELINE_PREDICT`   produces the forecast from the snapshot + future-covariate context.
  - A **skip-fit mode** lets you do both in a single SQL call: pass an empty   model table plus a unified `[ID, TARGET (NULL for future rows), COV...]`   table; the operator infers in one round trip.

<br><br>

### Understanding the exercise scenario 

In this exercise, you will explore how to apply SAP-RPT-TS to generate __bike sharing counts forecast predictions__ in SAP HANA Cloud

![](./images/ex1_scenario_overview.png)


- You will explore and familiarize with the Bike Sharing data using HANA dataframes and time series visualizations
- Next you apply SAP-RPT-TS using Pipeline skip-fit functions for direct forecasts with and without covariate data and review the forecast accuracy. 
- You apply the massive pipeline functions to generate zero-shot forecasts using SAP-RPT-TS for multiple Bike Sharing subsidiaries in parallel

  
<br>

**Use case dataset reference:**

Fanaee-T, H. (2013). Bike Sharing [Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). UCI Machine Learning Repository. https://doi.org/10.24432/C5W894.<br>


<br><br>

## Ex. 1.0 - Connect to your SAP HANA Cloud instance

### Step 0: Establish and check connection


Throughout this exercise, we will be using the __Python Machine Learning client for SAP HANA (hana-ml)__, as your reference to all its functions and capabilities see the [hana-ml documentation](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/2026_3_QRC/en-US/change_log.html). The current version released with SAP HANA Cloud 2026 Q3 release is 2.30.
- The python package hana-ml in general allows to script in python, while SQL code is generated on-the-fly and directly passed to a connected SAP HANA database system for execution.
    - It allows to access and prepare data by means of a HANA dataframe, a python object holding a SQL select query. Many methods are provided to be used with the HANA dataframe, changing the SQL select query behind the scenes.
    - As its core, it provides methods to apply AI functions (algorithms from the [Predictive Analysis Library (PAL)](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-predictive-analysis-library/sap-hana-cloud-sap-hana-database-predictive-analysis-library-pal?locale=en-US&version=LATEST) and [Automated Predictive Library (APL)](https://help.sap.com/docs/apl?locale=en-US&version=LATEST) to the data prepared with a HANA dataframe, designed to apply all the processing within the SAP HANA database.
- The python package is released as a component with every the SAP HANA Client delivery, in addition the latest __hana-ml__-version can always be found at the pypi public repository at https://pypi.org/project/hana-ml/.
<br><br>

In Python, installed packages require to be __imported__ to the current session, in order to be available for use in python scripts.
- Let's first run the import of hana-ml

In [ ]:
# Importing the Python Machine Learning client library for SAP HANA and get the version
import hana_ml
print(hana_ml.__version__)

Next, execute the next cell and the referenced script, prepared in the Getting Started section to connect to the SAP HANA CLoud database.

In [ ]:
%run "../ex0/ex0_connect2HANACloud.ipynb"

## Ex. 1.1 - Load and explore the Bike Sharing time series data

The Bike Sharing [Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) is pre-loaded to a SAP HANA Cloud table. A script example for loading the original data set into a SAP HANA Cloud table is given in the Appendix section. <br>
<br>
The dataset refers to hourly counts of rental bikes between years 2011 and 2012 of the Capital Bikeshare System with the corresponding weather and seasonal information. The hourly time series history includes 17389 rows and the following original columns <br>
- DATE_HOUR (original dteday) : date and hour <br>
- SEASON : season (spring, summer, fall, winter) <br>
- HOLIDAY : weather day is holiday or not  <br>
- WEEKDAY : day of the week <br>
- WORKINDAY : if day is neither weekend nor holiday is 1, otherwise is 0. <br>
- WEATHER: 'Clear, Few clouds' | 'Mist + Cloudy' | 'Light Snow, Light Rain' | 'Heavy Rain, Heavy Snow' <br>
- TEMP : Normalized temperature in Celsius. The values are divided to 41 (max) <br>
- ATEMP: Normalized feeling temperature in Celsius. The values are divided to 50 (max) <br>
- HUMIDITY: Normalized humidity. The values are divided to 100 (max) <br>
- WINDSPEED: Normalized wind speed. The values are divided to 67 (max) <br>
- CASUAL: count of casual users (column not used) <br>
- REGISTERED: count of registered users (column not used)<br>
- CNT: count of total rental bikes, our TARGET column to be forecasted <br>
 <br>

For the purpose of illustrating use of massive, data-parallel forecasting with SAP-RPT-TS and the SAP HANA Cloud massive pipeline methods, based on today's subsidiaries / affiliates of  Captial Bike Sharing (CBS) at Washington D.C. the same bike sharing time series data has been inserted into the database table for each of the today's neighboring cities and counties
 other cities <br>
-  Alexandria, VA (City) Virginia Jurisdictions, Falls Church VA (City), Fairfax VA (City), Arlington County VA, Fairfax County VA, Montgomery County  MD Maryland Jurisdictions. Prince George's County  MD Maryland Jurisdictions <br>
 <br>

### Step 1: Access the BikeSharing data and create HANA dataframes

A [SAP HANA dataframe](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/2026_3_QRC/en-US/hana_ml.dataframe.html) is a python object of __hana_ml__
- which holds a SQL select statements against data in SAP HANA Platform / Cloud
- can be created directly against a table or using a SQL statement, incl. multi-statement SQL block
- may included parameters, thus also Calculation Views and others can be consumed
- create a HANA dataframe __bike_all_hdf__ over the basis table in schema DA262_SHARED

In [ ]:
# Based on the connection object "myconn", using "table" the HANA dataframe is created
import hana_ml.dataframe as dataframe
bike_all_hdf=myconn.table("BIKESHARING", schema="DA262_SHARED")
bike_all_hdf.select_statement


<br>

Only when we make use of the __collect()__ method, the result-set of the dataframe SQL select statement is transferred into the python environment
- methods before the collect()-method, like sort, head, tail are executed beforehand, and thus may filter the result

In [ ]:
# Sorting and filtering the dataframe
display(bike_all_hdf.sort(['LOCATION', 'DATE_HOUR']).head(2).collect())
display(bike_all_hdf.sort(['LOCATION', 'DATE_HOUR']).tail(2).collect())
print()
display(bike_all_hdf.sort(['LOCATION', 'DATE_HOUR']).tail(2).select_statement)
print()
# Listing the HANA dataframe columns
print(bike_all_hdf.columns)

<br>

The __describe-method__ is a popular method overview the individual dataframe column statistics (unique, nulls, mean, ...)

In [ ]:
bike_all_hdf.describe().collect()

In [ ]:
# Let's just remind us what there Bike Sharing subsidiary locations in our data using the distinct-method
bike_all_hdf.distinct('LOCATION').collect()

In [ ]:
# Are the Bike Sharing counts different by location? 
agg_hdf = bike_all_hdf.agg([('sum', 'CNT', 'CNT_SUM')], group_by=['LOCATION'])
agg_hdf = agg_hdf.sort(['CNT_SUM'], desc=True)

display(agg_hdf.tail(3).collect())

<br>
<br>

__Create a HANA dataframe bike_hist_hdf, of only the bike sharing counts history__
- Therefore we are filtering the hourly data, for everything before '2012-12-10 23:59:00'
- Hence, we seek the forecast model to predict the 14 days following '2012-12-10 23:59:00'

In [ ]:
# Reodering the columns: 1. <key-column>, 2. <target-column>, 3. <all covariates columns>
bike_all_hdf=bike_all_hdf.to_head('CNT').to_head('DATE_HOUR').to_head('LOCATION')

# Dropping 2 columns, we are not interested in as part of this exercise
bike_all_hdf=bike_all_hdf.drop('CASUAL').drop('REGISTERED')

# Filtering by time
bike_hist_hdf=bike_all_hdf.filter('DATE_HOUR < TO_TIMESTAMP(\'2012-12-10 23:59:00\')')
display(bike_hist_hdf.sort(['LOCATION', 'DATE_HOUR']).tail(2).collect())

<br>

__Create a HANA dataframe bike_test_hdf, over 2 weeks of data to test the forecast accuracy__
- it holds the 14 days following '2012-12-10 23:59:00', for which we know the bike sharing counts

In [ ]:
bike_test_hdf=bike_all_hdf.filter('DATE_HOUR between TO_TIMESTAMP(\'2012-12-10 23:59:00\') and TO_TIMESTAMP(\'2012-12-22 06:59:00\')')
display(bike_test_hdf.sort(['LOCATION', 'DATE_HOUR']).tail(2).collect())

<br>
<br>

### Step 2: Further explore and visualize the time series data

<br>

Most interestingly however, how does the bike sharing data look over time using a simple plot method

In [ ]:
# Select the columns we need, into a temporary *plot-dataframe
hdf_data_plot = bike_all_hdf.select('LOCATION','DATE_HOUR', 'CNT').sort('DATE_HOUR') 

# Filtering the data to a single location
hdf_data_plot=hdf_data_plot.filter('"LOCATION" in (\'Washington, D.C.\')')

# Plot the data visually using a plotly-plot
import plotly.express as px
fig = px.line(hdf_data_plot.collect() , x='DATE_HOUR', y='CNT', title='Bike sharing counts in Washington, D.C.')
fig.show()

<br><br>

## Ex 1.2 - Forecast with "SAP-RPT-TS"

### Step 3: Predict without Covariates

_TSFM operator in AutoML and Pipeline__ functions 
- The hana_ml reference ([TSFM in hana_ml 2.30](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/2026_3_QRC/en-US/pal/topics/pipeline_operator_tsfm.html))
    - the TSFM operator can simply be called without any parameter (or arguments) as `{"TSFM": {"args": {},  "inputs": {"data": "ROWDATA"} } }`
<br>

- Key parameters (or arguments) to TSFM we may want ton include are 
    - `FORECAST_LENGTH`, Forecast the next 'n' time periods (max value 1024)
    - `POINT_FORECAST`,  Use the median or mean as the central forecast value 
    - `CLIP_MIN`, if set to 0, never predict negative CNT value

<br>

- data patterns / modes to use the TSFM operator
    - we can pass in covariates or not
    - we can pass the future rows with timepoints for which we expect forecasted values and TARGET=NULL or omit those and use FORECAST_LENGTH to set the forecast horizon
<br>
  
- in addition, the predict-method allows to enable prediction internal values
    - `confidence_level`, e.g. 0.9 sets 90% upper and lower bounds prediction interval
        - CONFIDENCE_LEVEL must be one of the specified values: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.98]

In [ ]:
# Defininig the pipeline and TSFM operator arguments
from hana_ml.algorithms.pal.pipeline import Pipeline
pipe = Pipeline(pipeline={"TSFM": {"args": {"POINT_FORECAST": ["median"], "CLIP_MIN": 0, "CLIP_MAX": 1500, "FORECAST_LENGTH": 271},   # "OUTPUT_QUANTILE": 0 
                                   "inputs": {"data": "ROWDATA"}}})

# Filtering the exact dataframe we seek to pass-in (we are also shortening the context a bit, max context values are 10000)
tmp_predict_step3_hdf=bike_hist_hdf.filter('"LOCATION" in (\'Washington, D.C.\') AND DATE_HOUR > TO_TIMESTAMP(\'2011-10-28 23:59:00\') ').select('DATE_HOUR', 'CNT').sort('DATE_HOUR')

result = pipe.predict(tmp_predict_step3_hdf, 
                      key='DATE_HOUR', 
                      output_prediction_interval=True, confidence_level=0.9)

<br>

What was actually executed as SQL code on the SAP HANA Cloud instance?

In [ ]:
print(pipe.get_predict_execute_statement())

In [ ]:
# Maximizing the row and column width for display
import pandas as pd
pd.set_option('max_colwidth', None) 

# Show the forecasted results, for the beginning and the end of the forecasted period (here, see FORECAST_LENGTH)
display(result.head(2).collect())
display(result.tail(2).collect())

# Show the forecasted CNT values, and prediction interval upper and lower bound values in seperate columns
predres_hdf=result.sort("ID").select(('TO_TIMESTAMP("ID")', 'DATE_HOUR'), 
                                     ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.value\'))', 'Forecast'),
                                     ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.lower\'))', 'LOWER_90%PI'), 
                                     ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.upper\'))', 'UPPER_90%PI'))
predres_hdf.head(10).collect()

<br>

__Show the result as a [forecast line plot](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/2026_3_QRC/en-US/hana_ml.visualizers.html)__

In [ ]:
# We want to show both, history values (actuals) and predicted values)
# Filtering the bike_hist_hdf-dataframe
group_key = 'Washington, D.C.'
tmp_actual_step3_hdf=bike_all_hdf.filter(f""" "LOCATION" = '{group_key}' and DATE_HOUR BETWEEN TO_TIMESTAMP(\'2012-11-26 23:59:00\') AND TO_TIMESTAMP(\'2012-12-22 06:59:00\') """)
tmp_actual_step3_hdf=tmp_actual_step3_hdf.select('DATE_HOUR', 'CNT')

# Build the forecast_line_plot
from hana_ml.visualizers.visualizer_base import forecast_line_plot
forecast_line_plot(pred_data=predres_hdf.set_index("DATE_HOUR"),  
                   confidence=("LOWER_90%PI", "UPPER_90%PI"),
                   actual_data=tmp_actual_step3_hdf.set_index("DATE_HOUR")
                   ) 

<br>

__Measure the forecast accuracy of the previous prediction__

In [ ]:
# Prepare a dataframe "accuracy_hdf", which includes Actuals and Forecast-values for the test period
forecast_hdf=predres_hdf.select('DATE_HOUR', 'Forecast').sort(['DATE_HOUR']) 
actuals_hdf=tmp_actual_step3_hdf.select('DATE_HOUR', 'CNT').sort(['DATE_HOUR']) 

accuracy_hdf=actuals_hdf.set_index(['DATE_HOUR']).join(forecast_hdf.set_index(['DATE_HOUR']), how='inner' )
display(accuracy_hdf.sort(['DATE_HOUR']) .rename_columns(['DATE_HOUR', 'Actual CNT', 'Forecasted CNT']).head(2).collect())
#display(accuracy_hdf.sort(['DATE_HOUR']) .rename_columns(['DATE_HOUR', 'Actual CNT', 'Forecasted CNT']).tail(2).collect())

In [ ]:
from hana_ml.algorithms.pal.tsa.accuracy_measure import accuracy_measure
res = accuracy_measure(data=accuracy_hdf.drop('DATE_HOUR'),
                                  evaluation_metric=['mse', 'rmse', 'mpe', 'et', 'mad', 'mase', 'wmape', 'smape', 'mape'])     # Forecast accuracy metrics
display(res.collect())
res.save('FCST_ACCURACY1', force=True)

<br>
<br>

### Step 4:  Predict with Covariates, incl. future values 

While in previous, the data we passed to SAP-RPT-TS
- only included history rows of CNT, no covariates
- and the forecast horizon was specified with the parameter "FORECAST_LENGTH"

__Now we want to take a different approach:__
- along with the CNT data, we __additionally pass-in covariate data__ to improve the forecast accuracy
- this includes __future values of covariates__
- the __forecast horiozon__ period will be determined by the __future time points__ passed in to SAP-RPT-TS for prediction


<br> <br>

Therefore, we __first__ create a HANA dataframe __bike_predict_hdf__ which includes
- the __known covariates__ for the complete 2 weeks: __'SEASON', 'MONTH', 'HOLIDAY', 'WEEKDAY', 'WORKINGDAY'__
- the __weather-related covariates__, are only known (forecasted itself) for the first week for the forecast: __'WEATHER', 'TEMP', 'ATEMP', 'HUMIDITY', 'WINDSPEED'__
- we set the __CNT value to NULL__
- the __future time points__ determine the __forecast horizon__

In [ ]:
# Creating temporary HANA dataframes for each of the weeks, and then unioning thos together again
# HANA dataframe for the first week
tmp1_hdf=bike_all_hdf.filter('DATE_HOUR BETWEEN TO_TIMESTAMP(\'2012-12-10 23:59:00\') and TO_TIMESTAMP(\'2012-12-17 23:59:00\')')
tmp1_hdf=tmp1_hdf.drop('CNT').select('*', ('TO_INT(NULL)', 'CNT')) # the target column shall be missing certainly for the predict dataframe

# HANA dataframe for the second week
tmp2_hdf=bike_all_hdf.filter('DATE_HOUR BETWEEN TO_TIMESTAMP(\'2012-12-17 23:59:00\') and TO_TIMESTAMP(\'2012-12-22 06:59:00\')')
tmp2_hdf=tmp2_hdf.select('LOCATION', 'DATE_HOUR', 'SEASON', 'MONTH', 'HOLIDAY', 'WEEKDAY', 'WORKINGDAY', 
                         ('NULL', 'WEATHER'), ('NULL', 'TEMP'), ('NULL', 'ATEMP'), ('NULL', 'HUMIDITY'), ('NULL', 'WINDSPEED'), ('NULL', 'CNT'))

# Union of both and reordering columns
bike_predict_hdf=tmp1_hdf.union(tmp2_hdf)
bike_predict_hdf=bike_predict_hdf.to_head('CNT').to_head('DATE_HOUR').to_head('LOCATION')

# Review the resulting dataframe, beginning and ending periods
display(bike_predict_hdf.sort(['LOCATION', 'DATE_HOUR']).head(2).collect())
display(bike_predict_hdf.sort(['LOCATION', 'DATE_HOUR']).tail(2).collect())


<br>

__Secondly__, we create a HANA dataframe __bike_fitpredict_hdf__ which unions
- the __history dataframe__ (bike_hist_hdf) of covariates and CNT
- the prepared __predict dataframe__ (bike_predict_hdf) with future covariates and target set to NULL

In [ ]:
# Union the history and the predict dataframes
bike_fitpredict_hdf=bike_hist_hdf.union(bike_predict_hdf)

display(bike_fitpredict_hdf.sort(['LOCATION', 'DATE_HOUR']).head(2).collect())
display(bike_fitpredict_hdf.sort(['LOCATION', 'DATE_HOUR']).tail(2).collect())

<br>

__Call SAP-RPT-TS to FORECAST with the TSFM operator in the Pipeline function__
- note, we are passing in the fitpredict-dataframe where the forecast horizon is determined by the future time points, where our target CNT is NULL

In [ ]:
# Filter the dataframe bike_fitpredict_hdf and a shorter context history
tmp_fitpredict_step4_hdf=bike_fitpredict_hdf.filter('"LOCATION" in (\'Washington, D.C.\') AND DATE_HOUR > TO_TIMESTAMP(\'2011-11-11 23:59:00\') ').deselect('LOCATION')

# Prepare and call the Pipeline-function to forecast using SAP-RPT-TS
from hana_ml.algorithms.pal.pipeline import Pipeline
pipe = Pipeline(pipeline={"TSFM": {"args": {"POINT_FORECAST": ["median"], "CLIP_MIN": 0, "CLIP_MAX": 1500},   
                                   "inputs": {"data": "ROWDATA"}}})

result = pipe.predict(tmp_fitpredict_step4_hdf.sort('DATE_HOUR'), 
                      key='DATE_HOUR', 
                      output_prediction_interval=True, confidence_level=0.9)

<br>

__Briefly checking the forecasted values and visualize the results with a Forecast Line Plot__

In [ ]:
predres_hdf=result.sort("ID").select(('TO_TIMESTAMP("ID")', 'DATE_HOUR'), 
                                     ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.value\'))', 'Forecast'),
                                     ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.lower\'))', 'LOWER_90%PI'), 
                                     ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.upper\'))', 'UPPER_90%PI'))
predres_hdf.head(5).collect()

In [ ]:
# Filtering the bike_all_hdf-dataframe for actuals
group_key = 'Washington, D.C.'
tmp_actual_step4_hdf=bike_all_hdf.filter(f""" "LOCATION" = '{group_key}' and DATE_HOUR BETWEEN TO_TIMESTAMP(\'2012-11-26 23:59:00\') AND TO_TIMESTAMP(\'2012-12-22 06:59:00\') """)
tmp_actual_step4_hdf=tmp_actual_step4_hdf.select('DATE_HOUR', 'CNT')

# Show actuals and forecasted values in the Forecast Line Plot
from hana_ml.visualizers.visualizer_base import forecast_line_plot
forecast_line_plot(pred_data=predres_hdf.set_index("DATE_HOUR"),  #.rename_columns({'TIME_STAMP': 'DATE'})
                   confidence=("LOWER_90%PI", "UPPER_90%PI"),
                   actual_data=tmp_actual_step4_hdf.set_index("DATE_HOUR"))

<br>


__Measure the forecast accuracy of the forecast with covariates__
- Could the forecast accuracy be improved by adding the covariates?

In [ ]:
# Prepare a dataframe, which includes Actuals and Forecast-values for the test period
forecast_hdf=predres_hdf.select('DATE_HOUR', 'Forecast').sort(['DATE_HOUR']) 
actuals_hdf=tmp_actual_step4_hdf.select('DATE_HOUR', 'CNT').sort(['DATE_HOUR']) 

accuracy_hdf=actuals_hdf.set_index(['DATE_HOUR']).join(forecast_hdf.set_index(['DATE_HOUR']), how='inner' )

In [ ]:
from hana_ml.algorithms.pal.tsa.accuracy_measure import accuracy_measure
res2 = accuracy_measure(data=accuracy_hdf.drop('DATE_HOUR'),
                                  evaluation_metric=['mse', 'rmse', 'mpe', 'et', 'mad', 'mase', 'wmape', 'smape', 'mape']) 

accuracy_comparison_hdf=res.set_index(['STAT_NAME']).join(res2.rename_columns({"STAT_VALUE":"STAT_VALUE with Covariates"}).set_index(['STAT_NAME']), how='inner' )
display(accuracy_comparison_hdf.collect())

<br>
<br>

## Ex. 1.3 - Forecast multiple time series with "SAP-RPT-TS"

### Step 5:  Forecast with multiple, independent time series with Covariates

<br>

The __bike_fitpredict_hdf__ dataframe, already includes the __LOCATION__ column, as the __identifyer for each independent time series__

In [ ]:
display(bike_fitpredict_hdf.sort('DATE_HOUR').head(8).collect())
display(bike_fitpredict_hdf.sort('DATE_HOUR').tail(8).collect())

<br>

Using __MassivePipeline__ method, allows to __generate forecasts using SAP-RPT-TS for each independent time series in parallel__

In [ ]:
# Filtering the fitpredict_hdf
tmp_step5_bike_fitpredict_hdf=bike_fitpredict_hdf.filter('DATE_HOUR > TO_TIMESTAMP(\'2011-11-11 23:59:00\')')

# Using MassivePipeline method, with group_key 'Location'
from hana_ml.algorithms.pal.massive_pipeline import MassivePipeline
mpipe = MassivePipeline(pipeline={"TSFM": {"args": { "POINT_FORECAST": ["median"], "CLIP_MIN": 0, "CLIP_MAX": 10000}, 
                                           "inputs": {"data": "ROWDATA"}}})

result=mpipe.predict(tmp_step5_bike_fitpredict_hdf, 
                     group_key='LOCATION', 
                     key='DATE_HOUR', 
                     output_prediction_interval=True, confidence_level=0.9)  
    
result.collect()

In [ ]:
# The error_ result dataframe would show if there are specific errors for any of the LOCATION time series
mpipe.error_.head(100).collect()

In [ ]:
# Inspect the SQL procedure call executed, uncomment the next row (delete #)
#print(mpipe.get_predict_execute_statement()) 

<br>

__Create a Forecast Line Plot for location "Arlington County, VA"__

In [ ]:
# Capture forecasted value, upper and lower bounds for forecast response
predres_hdf=result.sort("ID").select('LOCATION', ('TO_TIMESTAMP("ID", \'YYYY-MM-DD HH24:MI:SS.FF7\')', 'DATE_HOUR'), #, \'YYYY-MM-DD HH:MI:SS.MS\'
                                          ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.value\'))', 'Forecast'), 
                                          ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.lower\'))', 'LOWER_90%PI'), 
                                          ('TO_DOUBLE(JSON_VALUE("SCORES", \'$.upper\'))', 'UPPER_90%PI'))

# Filter for 
group_key = 'Arlington County, VA'
tmp_actual_step5_hdf=bike_all_hdf.filter('DATE_HOUR BETWEEN TO_TIMESTAMP(\'2012-11-26 23:59:00\') AND TO_TIMESTAMP(\'2012-12-22 06:59:00\')')

# Create the Forecast Line Plot
from hana_ml.visualizers.visualizer_base import forecast_line_plot
forecast_line_plot(pred_data=predres_hdf.filter(f""" "LOCATION" = '{group_key}'""").drop("LOCATION").set_index("DATE_HOUR"),
                   confidence=("LOWER_90%PI", "UPPER_90%PI"),
                   actual_data=tmp_actual_step5_hdf.filter(f""" "LOCATION" = '{group_key}'""").select('DATE_HOUR', 'CNT').set_index("DATE_HOUR"))

<br>

__Evaluate Forecast Accuracy for multiple time series__
- Note, there is also a massive-option (massive=True) for the accuracy_measure-function

In [ ]:
# Create the accuracy-input dataframe with Forecasted and Actual (CNT) values for each LOCATION
forecast_hdf=predres_hdf.select('LOCATION', 'DATE_HOUR', 'Forecast').sort(['LOCATION', 'DATE_HOUR'])
actuals_hdf=tmp_actual_step5_hdf.select('LOCATION', 'DATE_HOUR', 'CNT').sort(['LOCATION', 'DATE_HOUR'])

accuracy_hdf=actuals_hdf.set_index(['LOCATION', 'DATE_HOUR']).join(forecast_hdf.set_index(['LOCATION', 'DATE_HOUR']), how='inner' )
display(accuracy_hdf.head(5).collect())

# Calculate the accuracy for all locations
from hana_ml.algorithms.pal.tsa.accuracy_measure import accuracy_measure
res, error = accuracy_measure(data=accuracy_hdf.drop('DATE_HOUR'),
                                  massive=True,
                                  evaluation_metric=['mse', 'rmse', 'mpe', 'et',
                                              'mad', 'mase', 'wmape', 'smape',
                                              'mape'])

# Display accuracy results for 
group_key = 'Arlington County, VA'
display(res.filter(f""" "LOCATION" = '{group_key}'""").collect())
#display(error.collect())

# Appendix


## Ask an LLM to explain the forecast accuracy statistics

In [ ]:
forecast_explained_hdf = myconn.sql(
"""
WITH forecast_data AS (
    SELECT 
        STRING_AGG(STAT_NAME || ': ' || STAT_VALUE, CHAR(10)) AS stats_payload
    FROM 
        FCST_ACCURACY1
)
SELECT AI_TEXT_COMPLETION(
'You are an business user trying to understand forecast accuracy. Evaluate whether the provided forecasting metrics signify a good or bad forecast. Briefly summarize in 100 words the overall reasoning and judgement. Here are the statistical values from my forecast model:\n' || stats_payload,
'"gpt-4o-mini"', 
NULL,
'DA262_AICORE_ORCHESTRATION')
FROM forecast_data;
""")
forecast_explained_hdf.collect()

## Load Bike Sharing data

Once the dataset has been downloaded from the source, it can be loaded into the SAP HANA instance using the following script example
- Fanaee-T, H. (2013). Bike Sharing [Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). UCI Machine Learning Repository. https://doi.org/10.24432/C5W894.<br>

In [ ]:
import pandas as pd
df_data = pd.read_csv("./hour.csv", sep=",")
display(df_data.dtypes)

import hana_ml.dataframe as dataframe
upload_hdf = dataframe.create_dataframe_from_pandas(connection_context=myconn, 
                                                   pandas_df=df_data, 
                                                   table_name='bike_hour',
                                                   force=True,
                                                   replace=False)
upload_hdf.dtypes()
display(upload_hdf.sort(['dteday', 'hr']).head(2).collect())
display(upload_hdf.sort(['dteday', 'hr']).tail(2).collect())

The following SQL statements apply some changes to the dataset for the exercise.